In [ ]:
#Install libraries
!pip install pandas
!pip install numpy
!pip install scanpy

In [ ]:
#Import libraries
import pandas as pd
import numpy as np  
import scanpy as sc
import sys

In [ ]:
#Load in dataset
df = pd.read_csv('/path/to/your/file/GSE233866_untreated_counts.csv', index_col=0)

# Convert to AnnData (cells as rows!)
adata = sc.AnnData(df.T)

In [ ]:
#Annotate mitochondrial and ribosomal genes
# Mitochondrial genes (human: MT-, mouse: mt-)
adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')

# Ribosomal genes (RPL, RPS)
adata.var['ribo'] = adata.var_names.str.upper().str.startswith(('RPL', 'RPS'))

In [ ]:
#Compute QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo'], inplace=True)

In [ ]:
#Rename columns to match description
adata.obs['percent_mito'] = adata.obs['pct_counts_mt']
adata.obs['percent_ribo'] = adata.obs['pct_counts_ribo']

In [ ]:
#Cell cycle scoring 
#Loads standard Seurat gene lists manually 
s_genes = [
    'Mcm6', 'Exo1', 'Dtl', 'Cdca7', 'Rad51', 'Wdr76', 'Pcna', 'Pola1', 'Ccne2', 'Casp8ap2', 'Usp1', 'Nasp', 'Clspn', 'Rpa2', 'Tyms', 'Slbp', 'Ung', 'Rfc2', 'Mcm2', 'Rad51ap1', 'E2f8', 'Blm', 'Pold3', 'Rrm1', 'Prim1', 'Mcm5', 'Gins2', 'Tipin', 'Brip1', 'Cdc6', 'Gmnn', 'Rrm2', 'Ubr7', 'Dscc1', 'Atad2', 'Mcm4', 'Cdc45', 'Chaf1b', 'Uhrf1', 'Msh2', 'Fen1', 'Hells'
]

g2m_genes = [
    'Hjurp', 'Nuf2', 'Lbr', 'Cenpf', 'Nek2', 'Tubb4b', 'Ckap5', 'Nusap1', 'Bub1', 'Ckap2l', 'Tpx2', 'Ube2c', 'Aurka', 'Ect2', 'Smc4', 'Cks1b', 'Anp32e', 'Psrc1', 'Cenpe', 'Cdc20', 'Cdca8', 'Cenpa', 'Tacc3', 'Cdca3', 'Mki67', 'Cdk1', 'Gas2l3', 'Tmpo', 'Ckap2', 'Hmgb2', 'Ctcf', 'Anln', 'Kif23', 'Ccnb2', 'Ttk', 'Hmmr', 'Aurkb', 'Top2a', 'Birc5', 'Cks2', 'G2e3', 'Gtse1', 'Cbx5', 'Ndc80', 'Kif20b', 'Kif11'
]

sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

In [ ]:
#Compute relative expression per gene per cell
#Normalize per cell (UMI fraction)
adata.layers["relative"] = adata.X / adata.X.sum(axis=1, keepdims=True)

In [ ]:
#Check Malat1 expression
malat1_mask = adata.var_names.str.lower() == "malat1"

#Mean fraction across cells
malat1_fraction = np.mean(adata[:, malat1_mask].layers["relative"])

print("Mean Malat1 fraction:", malat1_fraction)

In [ ]:
#Remove Malat1
adata = adata[:, ~(adata.var_names.str.lower() == "malat1")]

In [ ]:
#Filter genes (present in ≥10 cells)
sc.pp.filter_genes(adata, min_cells=10)

In [ ]:
#Filter cells and libaries based on set thresholds
adata = adata[
    (adata.obs.n_genes_by_counts >= 500) &
    (adata.obs.n_genes_by_counts <= 10000) &
    (adata.obs.percent_mito < 5),
    :
]

In [ ]:
#Filter for dopaminergic cells
#Gene names to match (case-insensitive)
genes = ["Th", "Slc6a3"]

#Check availability (optional sanity check)
available = adata.var_names.str.lower()
for g in genes:
    print(g, (available == g.lower()).any())

#Get expression matrices for each gene
th = adata[:, adata.var_names.str.lower() == "th"].X
slc6a3 = adata[:, adata.var_names.str.lower() == "slc6a3"].X

#Convert to boolean expression (> 0)
th_expr = np.array(th > 0).flatten()
slc6a3_expr = np.array(slc6a3 > 0).flatten()

#Keep cells expressing either gene (OR condition)
keep = th_expr | slc6a3_expr

#Subset AnnData object
adata_filtered = adata[keep, :].copy()

#Check result
print(adata_filtered.shape)

In [ ]:
adata_filtered.write("QC_AnnDataObject.h5ad")